# Week 2 Day 2 — Pandas Advanced

**Topics:** `merge`, join types, `pivot_table`, `melt`, reshaping, and method chaining.

Dataset: Week 1 Day 5 sales data.


In [1]:
import pandas as pd
from pathlib import Path

sales = pd.read_csv("../../week-01/day-05/sales_data.csv")
sales["sale_date"] = pd.to_datetime(sales["sale_date"])

print("Sales shape:", sales.shape)
display(sales.head())


Sales shape: (10, 8)


,sale_id,customer_name,product,category,region,amount,sale_date,discount
0,1,Ananya,Laptop,Electronics,South,55000.0,2026-09-01,0.10
1,2,Rahul,Phone,Electronics,North,25000.0,2026-09-02,0.05
2,3,Priya,Chair,Furniture,West,4500.0,2026-09-03,NaN
3,4,Arjun,Desk,Furniture,South,8000.0,2026-09-04,0.00
4,5,Meera,Headphones,Electronics,East,2000.0,2026-09-05,0.15


## 1. Create two DataFrames and demonstrate all merge types

In [2]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "customer_name": ["Asha", "Bharat", "Charan", "Divya"]
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104],
    "customer_id": [1, 2, 2, 5],
    "amount": [1200, 850, 430, 990]
})

inner_join = pd.merge(customers, orders, on="customer_id", how="inner")
left_join = pd.merge(customers, orders, on="customer_id", how="left")
right_join = pd.merge(customers, orders, on="customer_id", how="right")
outer_join = pd.merge(customers, orders, on="customer_id", how="outer", indicator=True)

print("INNER JOIN")
display(inner_join)
print("LEFT JOIN")
display(left_join)
print("RIGHT JOIN")
display(right_join)
print("FULL OUTER JOIN")
display(outer_join)


INNER JOIN


,customer_id,customer_name,order_id,amount
0,1,Asha,101,1200
1,2,Bharat,102,850
2,2,Bharat,103,430


LEFT JOIN


,customer_id,customer_name,order_id,amount
0,1,Asha,101.0,1200.0
1,2,Bharat,102.0,850.0
2,2,Bharat,103.0,430.0
3,3,Charan,NaN,NaN
4,4,Divya,NaN,NaN


RIGHT JOIN


,customer_id,customer_name,order_id,amount
0,1,Asha,101,1200
1,2,Bharat,102,850
2,2,Bharat,103,430
3,5,NaN,104,990


FULL OUTER JOIN


,customer_id,customer_name,order_id,amount,_merge
0,1,Asha,101.0,1200.0,both
1,2,Bharat,102.0,850.0,both
2,2,Bharat,103.0,430.0,both
3,3,Charan,NaN,NaN,left_only
4,4,Divya,NaN,NaN,left_only
5,5,NaN,104.0,990.0,right_only


**Key point:** `merge()` combines rows using matching key columns. INNER keeps matches, LEFT keeps every left row, RIGHT keeps every right row, and OUTER keeps keys from both tables.

## 2. Pivot table — sales by region and product

In [3]:
pivot = pd.pivot_table(
    sales,
    index="region",
    columns="product",
    values="amount",
    aggfunc="sum",
    fill_value=0,
    margins=True
)

display(pivot)


product,Bookshelf,Chair,Desk,Headphones,Laptop,Monitor,Notebook,Pen,Phone,Tablet,All
region,,,,,,,,,,,
East,0.0,0.0,0.0,2000.0,0.0,0.0,0.0,0.0,0.0,18000.0,20000.0
North,0.0,0.0,0.0,0.0,0.0,0.0,150.0,0.0,25000.0,0.0,25150.0
South,6500.0,0.0,8000.0,0.0,55000.0,12000.0,0.0,0.0,0.0,0.0,81500.0
West,0.0,4500.0,0.0,0.0,0.0,0.0,0.0,50.0,0.0,0.0,4550.0
All,6500.0,4500.0,8000.0,2000.0,55000.0,12000.0,150.0,50.0,25000.0,18000.0,131200.0


## 3. Melt — wide format to long format

In [4]:
wide_sales = sales.groupby("region", as_index=False).agg(
    total_amount=("amount", "sum"),
    average_amount=("amount", "mean")
)

print("Wide format")
display(wide_sales)

long_sales = pd.melt(
    wide_sales,
    id_vars=["region"],
    value_vars=["total_amount", "average_amount"],
    var_name="metric",
    value_name="value"
)

print("Long format")
display(long_sales)


Wide format


,region,total_amount,average_amount
0,East,20000.0,10000.0
1,North,25150.0,12575.0
2,South,81500.0,20375.0
3,West,4550.0,2275.0


Long format


,region,metric,value
0,East,total_amount,20000.0
1,North,total_amount,25150.0
2,South,total_amount,81500.0
3,West,total_amount,4550.0
4,East,average_amount,10000.0
5,North,average_amount,12575.0
6,South,average_amount,20375.0
7,West,average_amount,2275.0


## 4. Reshape with pivot and pivot_table

In [5]:
region_metrics = sales.groupby(
    ["region", "category"], as_index=False
).agg(total_sales=("amount", "sum"))

reshaped = region_metrics.pivot(
    index="region",
    columns="category",
    values="total_sales"
)

display(reshaped)


category,Electronics,Furniture,Stationery
region,,,
East,20000.0,NaN,NaN
North,25000.0,NaN,150.0
South,67000.0,14500.0,NaN
West,NaN,4500.0,50.0


## 5. Method chaining — clean and summarize in one pipeline

In [6]:
summary = (
    pd.read_csv("../../week-01/day-05/sales_data.csv")
      .assign(sale_date=lambda df: pd.to_datetime(df["sale_date"]))
      .dropna(subset=["region", "category", "amount"])
      .query("amount > 0")
      .groupby(["region", "category"], as_index=False)
      .agg(
          total_sales=("amount", "sum"),
          average_sale=("amount", "mean"),
          sale_count=("amount", "size")
      )
      .sort_values("total_sales", ascending=False)
      .reset_index(drop=True)
)

display(summary)


,region,category,total_sales,average_sale,sale_count
0,South,Electronics,67000.0,33500.0,2
1,North,Electronics,25000.0,25000.0,1
2,East,Electronics,20000.0,10000.0,2
3,South,Furniture,14500.0,7250.0,2
4,West,Furniture,4500.0,4500.0,1
5,North,Stationery,150.0,150.0,1
6,West,Stationery,50.0,50.0,1


## Viva notes

1. **`merge()` vs `concat()`:** `merge()` matches rows using key columns; `concat()` stacks DataFrames along rows or columns.
2. **`melt()`:** Converts wide columns into identifier and value rows; useful when preparing data for analysis or visualisation.
3. **Pandas pivot table vs Excel PivotTable:** Both summarize data by categories. Pandas pivot tables are code-driven, repeatable, and integrate with Python data pipelines.

## Completion checklist

- [x] INNER, LEFT, RIGHT, and OUTER merges
- [x] Sales pivot table by region and product
- [x] Wide-to-long reshape using `melt()`
- [x] Additional reshape using `pivot()`
- [x] Method chaining
